In [68]:
import pandas as pd
from pathlib import Path
import re
from difflib import SequenceMatcher
from unidecode import unidecode
from scipy.optimize import linear_sum_assignment
import numpy as np

MAIN_FOLDER = r"C:\Users\L14\Downloads\ABH CSV"

#path = r"C:\Users\L14\Documents\SITUATION GLOBALE\CONTROLE_02_02_2026\CONTROLE CTB REMARQUES_total_fusionne.xlsx"

path_maj_gestionnaire = fr"{MAIN_FOLDER}\DEMANDES_total_fusionne.xlsx"
path_voisins_pvcl = fr"{MAIN_FOLDER}\VOISINS_PVCL_total_fusionne.xlsx"
path_voisins_signes = fr"{MAIN_FOLDER}\VOISINS_PRESENCE_total_fusionne.xlsx"
path_etat_voisin = fr"{MAIN_FOLDER}\Etat_tonkpi_15-03-26_Liv1_avec_Signature.xlsx"

paths_rem = Path(fr"{MAIN_FOLDER}\ALL_CSV\CONTROLE CTB REMARQUES")
paths_dig = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG")
paths_ctb = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB")

out_add_to_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\ADD_TO_CTB")
out_no_mask_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\NO_MASK")
out_replace_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\REPLACE_CTB")
out_dig_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG")
out_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB")
out_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\UPDATES_FILES")
out_to_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB\ADD_NEIGHBOR\TO_CTB")
out_ctb_manual_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB\ADD_NEIGHBOR\MANUAL")


paths_action_dig = paths_dig.glob("*.csv")
paths_action_ctb = paths_ctb.glob("*.csv")
CSVs = paths_dig.glob("*.csv")
files_rem = paths_rem.rglob("*.xlsx")

df_maj_gestionnaire = pd.read_excel(path_maj_gestionnaire, engine='openpyxl')
etat_voisins = pd.read_excel(path_etat_voisin, engine='openpyxl')
df_voisins_signes = pd.read_excel(path_voisins_signes, engine='openpyxl')
df_voisins_signes_ctrl = pd.read_excel(path_voisins_signes, engine='openpyxl')
df_voisins_pvcl = pd.read_excel(path_voisins_pvcl, engine='openpyxl')
#controle_voisins = pd.read_excel(path, engine='openpyxl')

df_maj_gestionnaire.loc[:, "village"] = (
    df_maj_gestionnaire["village"]
      .apply(lambda x: unidecode(x) if pd.notna(x) else x)
      .str.lower()
      .str.strip()
    )

df_maj_gestionnaire.loc[:, "code_par"] = df_maj_gestionnaire.loc[:, "village"] + "-" + df_maj_gestionnaire.loc[:, "numParcelleOF_c"]

#df_maj_gestionnaire.set_index("code_par", inplace=True)

etat_voisins["code_voisin_unique"] = (etat_voisins["indice_unique_voisin"].astype(str) + "__" + etat_voisins["num_demande"].astype(str))
etat_voisins = etat_voisins[etat_voisins["num_demande"].notna()]
etat_voisins_duplicated = etat_voisins[etat_voisins["code_voisin_unique"].duplicated()]
etat_voisins = etat_voisins.drop_duplicates(subset="code_voisin_unique")

df_voisins_signes["nameOfPerson"] = df_voisins_signes["nameOfPerson"].astype(str)
df_voisins_signes["numberCNI"] = df_voisins_signes["numberCNI"].astype(str)
df_voisins_signes["signatoryPhoto"] = df_voisins_signes["signatoryPhoto"].astype(str)
df_voisins_signes.loc[:, "village"] = (df_voisins_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )
df_voisins_signes.loc[:, "code_par"] = df_voisins_signes.loc[:, "village"] + "-" + df_voisins_signes.loc[:, "numParcelleOF"]
df_voisins_signes.loc[:, "code_vois"] = (
    df_voisins_signes["codePresence"].astype(str).str.strip()
    + "-"
    + df_voisins_signes["nameOfPerson"].astype(str).str.strip()
)
df_voisins_signes = (
    df_voisins_signes
    .drop_duplicates("code_vois", keep="first")
)
#df_voisins_signes.set_index("code_par", inplace=True)


df_voisins_signes_ctrl.loc[:, "village"] = (df_voisins_signes_ctrl["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )

df_voisins_pvcl.loc[:, "village"] = (df_voisins_signes_ctrl["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )
df_voisins_pvcl.loc[:, "code_par"] = df_voisins_pvcl.loc[:, "code"] + "-" + df_voisins_pvcl.loc[:, "numParcelleOF"]

df_voisins_pvcl.loc[:, "code_vois"] = (
    df_voisins_pvcl["code"].astype(str).str.strip()
    + "-"
    + df_voisins_pvcl["nameOfNeighbor"].astype(str).str.strip()
)

df_voisins_pvcl = (
    df_voisins_pvcl
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_signes_ctrl.loc[:, "code_par"] = df_voisins_signes_ctrl.loc[:, "codePresence"] + "-" + df_voisins_signes_ctrl.loc[:, "numParcelleOF"]

df_voisins_signes_ctrl.loc[:, "code_vois"] = (
    df_voisins_signes_ctrl["codePresence"].astype(str).str.strip()
    + "-"
    + df_voisins_signes_ctrl["nameOfPerson"].astype(str).str.strip()
)

df_voisins_signes_ctrl = (
    df_voisins_signes_ctrl
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_signes_ctrl.set_index("code_vois", inplace=True)


In [69]:
### Generate the file actions_digifor and actions_ctb to update data

#print(etat_voisins_duplicated.shape)
#print(etat_voisins_duplicated["code_voisin_unique"].head(10))

assert etat_voisins["code_voisin_unique"].is_unique, "❌ Le code voisin unique ne l'est pas"
etat_voisins.set_index("code_voisin_unique", drop=False, inplace=True)

def nearest_text_match(target, candidates, target_type='dig', min_score=0.5):
        """
        target : str (voisin DIGIFOR)
        candidates : list[str] (voisins CTB)
        """
        best = None
        unmatched = set(candidates)
        best_score = 0

        for c in candidates:
            if ("Aucun voisin" in target) or ("Aucun voisin" in c):
                    score = text_similarity(target, c)
            elif target_type == "dig":
                #print(c)
                score = text_similarity(target.split('(')[0], c.split(':')[-1].split('(')[0])
            elif target_type == "ctb":
                score = text_similarity(target.split(':')[-1].split('(')[0], c.split('(')[0])
            else:
                score = text_similarity(target, c)

            if score > best_score:
                best = c
                best_score = score
            
        if best_score >= min_score:
            unmatched -= {best}
            return best, unmatched, best_score

        return None, unmatched, best_score

def best_match_nn(digs, ctbs, min_score=0.6):
    if not digs or not ctbs:
        return [], digs.copy(), ctbs.copy()

    matrix = np.zeros((len(digs), len(ctbs)))

    for i, d in enumerate(digs):
        for j, c in enumerate(ctbs):
            if ("Aucun voisin" in d) or ("Aucun voisin" in c):
                score = text_similarity(d, c)
            else:
                score = text_similarity(d.split('(')[0], c.split(':')[-1].split('(')[0])
            matrix[i, j] = score

    row_ind, col_ind = linear_sum_assignment(-matrix)

    matches = []
    used_dig = set()
    used_ctb = set()

    for i, j in zip(row_ind, col_ind):
        score = matrix[i, j]
        if score >= min_score:
            matches.append((digs[i], ctbs[j], score))
            used_dig.add(digs[i])
            used_ctb.add(ctbs[j])

    unmatched_dig = [d for d in digs if d not in used_dig]
    unmatched_ctb = [c for c in ctbs if c not in used_ctb]

    return matches, unmatched_dig, unmatched_ctb

def split_clean(value):
    if pd.isna(value) or value == "RAS":
        return []
    return [v.strip() for v in value.split(",") if v.strip()]

def is_par(v): return re.match(r"^par\d+", v)
def is_riv(v): return re.match(r"^riv\d+", v)
def is_tit(v): return re.match(r"^tit\d+", v)
def is_zone(v): return re.match(r"^zone\d+", v)
def is_det(v): return "det_poly" in v
def is_signe(v): return "(signe)" in v or "(repr)" in v or "(non_par)" in v

def text_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def get_voisin_info(v, row_code):
    code_v = v.split(':')[0] + "__" + row_code
    if code_v in etat_voisins.index:
        r = etat_voisins.loc[code_v]
        code_voisin = r["num_plle_voisin"]
        subset = etat_voisins.loc[etat_voisins["num_plle"] == code_voisin]
        num_voisin = subset.iloc[0]["num_demande"] if not subset.empty else None
        return num_voisin, r["type_voisin"], r["orientation"]
    return None, None, None

def compare_names(row, code):
    
    rows_dig = []
    rows_ctb = []

    voisins_dig = split_clean(row["A corriger sur CTB ou Supprimer sur DIGIFOR"])
    voisins_ctb = split_clean(row["A ajouter ou Corriger sur DIGIFOR"])

    taille_dig = len(voisins_dig)
    taille_ctb = len(voisins_ctb)

    def add_row(old, new,code_voisin,type_voisin, position, action, comment):
        rows_dig.append({
            "code": row["code"],
            "code_parcelle": row["numParcelleOF"],
            "village": row["village"],
            "date": row["requestDate"],
            "ancien_nom": old,
            "nouveau_nom": new,
            "code_voisin": code_voisin,
            "type_voisin": type_voisin,
            "position": position,
            "action": action,
            "commentaire": comment
        })
    
    def add_row_ctb(old, new, type_voisin, position, action, comment):
        rows_ctb.append({
            "code": row["code"],
            "code_parcelle": row["numParcelleOF"],
            "village": row["village"],
            "date": row["requestDate"],
            "ancien_nom": old,
            "nouveau_nom": new,
            "type_voisin": type_voisin, 
            "position": position,
            "action": action,
            "commentaire": comment
        })

    if voisins_dig == ["Aucun voisin dans le pvcl presence"] or voisins_ctb == ["Aucun voisin dans le ctb"]:
        return [], []
    
    # 1️⃣ DIGIFOR RAS
    elif not voisins_dig and voisins_ctb:
        for v in voisins_ctb:
            code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, type_voisin, position, "ADD", "Ajout depuis CTB (DIGIFOR = RAS)")

    # 2️⃣ CTB RAS
    elif not voisins_ctb and voisins_dig:
        for v in voisins_dig:
            add_row(v, None, "", "", "", "DELETE", "Suppression (CTB = RAS)")

    # 3️⃣ 1–1
    elif taille_ctb == taille_dig == 1:
        ctb, dig = voisins_ctb[0], voisins_dig[0]
        code_voisin, type_voisin, position = get_voisin_info(ctb, row["code"])
        if is_par(ctb) or is_zone(ctb):
            add_row(None, ctb, code_voisin, type_voisin, position, "ADD", "Ajouter le voisin ctb = (par,zone) 1–1")
            add_row(dig, None, "", "", "", "DELETE", "Supprimer le voisin dig, ctb = (par,zone) 1–1")

        elif is_signe(ctb):
            ctb_new = ctb.split(':')[0] + ":" + dig
            code_voisin, type_voisin, position = get_voisin_info(ctb, row["code"])
            add_row(dig, ctb, code_voisin, type_voisin, position, "REPLACE_D",
                        f"Voisin a signe et a une corresponde sur CTB (score={score:.2f})")
        else:
            ctb_new = ctb.split(':')[0] + ":" + dig
            code_voisin, type_voisin, position = get_voisin_info(ctb, row["code"])
            add_row_ctb(ctb, ctb_new, type_voisin, position, "REPLACE_C", "Priorité au (signe) 1-1")

    # 4️⃣ 1 DIG vs N CTB
    elif taille_dig == 1:
        dig = voisins_dig[0]

        if all((is_par(v) or is_zone(v)) for v in voisins_ctb):
            for v in voisins_ctb:
                code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
                add_row(None, v, code_voisin, type_voisin, position, "ADD", "Ajout de voisin all(voisin par,,zone)")
            add_row(dig, None, "", "", "", "DELETE", "Suppression ancien voisin (all ctb =par,,zone)")

        else:
            best, unmatched, score = nearest_text_match(dig, voisins_ctb)
            par_zone = [v for v in unmatched if (is_par(v) or is_zone(v))]
            riv = [v for v in unmatched if not (is_par(v) or is_zone(v))]
            if best:
                ctb_new = best.split(':')[0] + ":" + best
                code_voisin, type_voisin, position = get_voisin_info(best, row["code"])
                if is_par(best) or is_zone(best):   
                    add_row(dig, best, code_voisin, type_voisin, position, "REPLACE_D",
                        f"Voisin a signe et a une corresponde sur CTB (score={score:.2f})")
                else:
                    add_row_ctb(best, ctb_new, type_voisin, position, "REPLACE_C",
                        f"Voisin a signe et a une corresponde sur CTB (score={score:.2f})")
                for v in unmatched:
                    code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
                    if is_par(v) or is_zone(v):
                        add_row(None, v, code_voisin, type_voisin, position, "ADD", "Ajout de voisin (voisin par or det)")
                    else:
                        add_row(None, v, code_voisin, type_voisin, position, "ADD_SEARCH", "Ajout de voisin (cherche signature)") 
            else:
                for v in par_zone:
                    code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
                    add_row(None, v, code_voisin, type_voisin, position, "ADD", "Ajout de voisin (voisin par or det or tit or zone)")
                ctb_new = riv[0].split(':')[0] + ":" + dig
                code_voisin, type_voisin, position = get_voisin_info(riv[0], row["code"])
                add_row_ctb(riv[0], ctb_new, type_voisin, position,
                    "MANUAL", "Remplacer par celui qui a signe")
                for v in riv[1:]:
                    add_row(None, v, "", "voisin_riv", position,
                    "ADD_SEARCH", "ajouter sur DIGIFOR et chercher la signature")

    # 5️⃣ 1 CTB vs N DIG
    elif taille_ctb == 1:
        ctb = voisins_ctb[0]
        code_voisin, type_voisin, position = get_voisin_info(ctb, row["code"])
        if is_par(ctb) or is_zone(ctb):
            add_row(None, ctb, code_voisin, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det)")
            for v in voisins_dig:
                add_row(v, None, "", "", "", "DELETE", "Nettoyage DIGIFOR")

        else:
            best, unmatched, score = nearest_text_match(ctb, voisins_dig, target_type='ctb')
            
            if best:
                ctb_new = ctb.split(':')[0] + ":" + best
                code_voisin, type_voisin, position = get_voisin_info(ctb, row["code"])
                add_row_ctb(ctb, ctb_new, type_voisin, position, "REPLACE_C",
                    f"Remplacer par un voisin qui a signe (score={score:.2f})")
                for v in unmatched:
                    add_row(v, None, "", "", "", "DELETE", "Nettoyage DIGIFOR")
            else:
                code_voisin, type_voisin, position = get_voisin_info(ctb, row["code"])
                
                for v in voisins_dig:
                    add_row(v, ctb,code_voisin, type_voisin, position,
                    "MANUAL", f"ajouter sur DIGIFOR et chercher la signature (se repete {len(voisins_dig)} fois)")
                
    # 6️⃣ N–N
    else:
        if all((is_par(v)  or is_zone(v)) for v in voisins_ctb):
            for v in voisins_ctb:
                code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
                add_row(None, v, code_voisin, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det ou tit ou zone)")
            for v in voisins_dig:
                add_row(v, None, "", "", "", "DELETE", "Nettoyage DIGIFOR all(voisin ctb = par ou det ou tit ou zone)")

        else:
            matches, rest_dig, rest_ctb = best_match_nn(
                voisins_dig, voisins_ctb
            )

            for d, c, score in matches:
                    if is_par(c) or is_zone(c):
                        code_voisin, type_voisin, position = get_voisin_info(c, row["code"])
                        add_row(d, c, code_voisin, type_voisin, position, "REPLACE_D",
                            f"Remplacer les correspondances ctb (par or det) et digifor (signe) (score={score:.2f})")
                    else:
                        ctb_new = c.split(':')[0] + ":" + d
                        code_voisin, type_voisin, position = get_voisin_info(c, row["code"])
                        add_row_ctb(c, ctb_new, type_voisin, position, "REPLACE_C",
                            f"Remplacer les correspondances ctb (riv) et digifor (signe) (score={score:.2f})")
            
            par_zone = [c for c in rest_ctb if (is_par(c) or is_zone(c))]
            riv = [c for c in rest_ctb if not (is_par(c) or is_zone(c))]

            for v in par_zone:
                code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
                add_row(None, v, code_voisin, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det ou tit ou zone)")
            
            for d, c in zip(rest_dig, riv):
                    ctb_new = c.split(':')[0] + ":" + d
                    code_voisin, type_voisin, position = get_voisin_info(c, row["code"])
                    add_row_ctb(c, ctb_new, type_voisin, position, 
                    "MANUAL", "Remplacer par celui qui a signe")
            
            if len(rest_dig) < len(riv):
                for v in riv[len(rest_dig)::]:
                    code_voisin, type_voisin, position = get_voisin_info(v, row["code"])
                    add_row(None, v, code_voisin, type_voisin, position,
                "ADD_SEARCH", "ajouter sur DIGIFOR et chercher la signature")
            elif len(rest_dig) > len(riv):
                for v in rest_dig[len(riv)::]:
                    add_row(v, None , "", "voisin_riv", "",
                "DELETE", "Plus de voisins sur DIGIFOR")

    return rows_dig, rows_ctb

for file in files_rem:
    print(file)
    controle_voisins = pd.read_excel(file, engine='openpyxl')
    all_rows_dig = []
    all_rows_ctb = []
    
    for idx, row in controle_voisins.iterrows():
        result_dig, result_ctb = compare_names(row, code=idx)
        all_rows_dig.extend(result_dig)
        all_rows_ctb.extend(result_ctb)

    result_df_dig = pd.DataFrame(all_rows_dig,
                            columns=["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "type_voisin", "code_voisin", "position", "action", "commentaire"])
    #result_df_dig["id"] = result_df_dig.index
    result_df_ctb = pd.DataFrame(all_rows_ctb,
                            columns=["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "type_voisin", "position", "action", "commentaire"])
    #result_df_ctb["id"] = result_df_ctb.index
    result_df_dig.to_csv(f"{out_dig_folder}/{file.name.split('.')[0]}_actions_digifor.csv",
                    index=False,
                    sep=";",
                    encoding="utf-8-sig")
    result_df_ctb.to_csv(f"{out_ctb_folder}/{file.name.split('.')[0]}_actions_ctb.csv",
                    index=False,
                    sep=";",
                    encoding="utf-8-sig")

C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT Aboudou 14-03_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT bossoh_copie_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT Calv 14-03_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT DIOMANDE PATRICK_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT Gompou 14-03_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT Gueu Kin 14-03_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT GUILLAUME_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT Hermann_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT JC 14-03_ctb.xlsx
C:\Users\L14\Downloads\ABH CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT Kader _ct

In [59]:
# --- Le but est de recuperer les signatures, cni dans le village pour les voisins à ajouter dans DIGIFOR ---

df_voisins_signes["code_vois"] = (
    df_voisins_signes["village"].astype(str).str.strip()
    + "-"
    + df_voisins_signes["nameOfPerson"].astype(str).str.strip()
)

df_voisins_pvcl["code_vois"] = (
    df_voisins_pvcl["village"].astype(str).str.strip()
    + "-"
    + df_voisins_pvcl["nameOfNeighbor"].astype(str).str.strip()
)
df_voisins_pvcl = (
    df_voisins_pvcl
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_signes = (
    df_voisins_signes
    .drop_duplicates("code_vois", keep="first")
)
# ------------------------------------------------
# Traitement des fichiers actions_ctb
# ------------------------------------------------

for path_action_dig in paths_action_dig:

    print("Processing:", path_action_dig)

    df_action_dig = pd.read_csv(path_action_dig, sep=";", encoding="utf-8-sig")

    # --- Nettoyage village
    df_action_dig["village"] = (
        df_action_dig["village"]
        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
        .str.lower()
        .str.strip()
    )

    # --- Masks
    mask_parcelle = df_action_dig["code_voisin"].str.contains(r"^P\d+", na=False)
    mask_riv = (df_action_dig["type_voisin"] == "voisin_riv") | (df_action_dig["action"] == "ADD")
    mask_delete = df_action_dig["action"].isin(["DELETE", "REPLACE_D"])

    # --- Nettoyage code parcelle
    df_action_dig.loc[mask_parcelle, "code_voisin"] = (
        df_action_dig.loc[mask_parcelle, "code_voisin"]
        .astype(str)
        .str.replace(r"(^P\d+)[A-Za-z]+", r"\1", regex=True)
    )

    # --- Création code_par
    df_action_dig.loc[mask_parcelle, "code_par"] = (
        df_action_dig.loc[mask_parcelle, "village"]
        + "-"
        + df_action_dig.loc[mask_parcelle, "code_voisin"]
    )

    # --- Nettoyage noms
    nom_nouveau = (
        df_action_dig["nouveau_nom"]
        .astype(str)
        .str.split(":", n=1)
        .str[-1]
        .str.strip()
    )

    df_action_dig.loc[mask_riv, "nom_a_ajouter"] = nom_nouveau
    df_action_dig.loc[mask_riv, "name"] = nom_nouveau

    df_action_dig.loc[mask_delete, "name"] = (
        df_action_dig.loc[mask_delete, "ancien_nom"]
        .astype(str)
        .str.split("(")
        .str[0]
        .str.strip()
    )

    # --- Création code_vois
    df_action_dig["code_vois"] = (
        df_action_dig["village"]
        + "-"
        + df_action_dig["name"]
            .fillna("")
            .astype(str)
            .str.split("(")
            .str[0]
            .str.strip()
    )

    # ------------------------------------------------
    # MERGE signatures voisins
    # ------------------------------------------------

    df_action_dig = df_action_dig.merge(
        df_voisins_signes[
            ["code_vois", "numberCNI", "signatoryPhoto", "nameOfPersonOriginal"]
        ],
        on="code_vois",
        how="left",
        suffixes=("_ctb","_dig")
    )

    df_action_dig.rename(
        columns={
            "numberCNI": "cni_voisin",
            "signatoryPhoto": "signature_voisin"
        },
        inplace=True
    )

    # ------------------------------------------------
    # MERGE PVCL ['nameOfPersonOriginal', 'nameOfNeighborOriginal', 'position_ctb', 'position_dig']
    # ------------------------------------------------

    df_action_dig = df_action_dig.merge(
        df_voisins_pvcl[
            ["code_vois", "position", "description","nameOfNeighborOriginal"]
        ],
        on="code_vois",
        how="left",
        suffixes=("_ctb","_dig")
    )

    # ------------------------------------------------
    # MERGE applicantNumber
    # ------------------------------------------------

    df_action_dig = df_action_dig.merge(
        df_maj_gestionnaire[
            ["applicantNumber", "identityDocumentNumber_x", "identityDocumentPhoto"]
        ],
        left_on="code_voisin",
        right_on="applicantNumber",
        how="left"
    )

    df_action_dig.rename(
        columns={"applicantNumber": "num_voisin"},
        inplace=True
    )

    # --- fallback CNI
    df_action_dig["cni_voisin"] = (
        df_action_dig["cni_voisin"]
        .replace("", pd.NA)
        .fillna(df_action_dig["identityDocumentNumber_x"])
    )

    # --- fallback signature
    df_action_dig["signature_voisin"] = (
        df_action_dig["signature_voisin"]
        .replace("", pd.NA)
        .fillna(df_action_dig["identityDocumentPhoto"])
    )

    # ------------------------------------------------
    # Nettoyage colonnes
    # ------------------------------------------------

    df_action_dig.drop(
        columns=[
            "identityDocumentNumber_x",
            "identityDocumentPhoto",
            "code_vois"
        ],
        errors="ignore",
        inplace=True
    )

    df_action_dig.reset_index(drop=True, inplace=True)

    # ------------------------------------------------
    # DEBUG
    # ------------------------------------------------

    #print("Total actions:", len(df_action_dig))
    #print("CNI trouvées:", df_action_dig["cni_voisin"].notna().sum())

    # ------------------------------------------------
    # Export
    # ------------------------------------------------

    df_action_dig.to_csv(
        path_action_dig,
        sep=";",
        encoding="utf-8-sig",
        index=False
    )

Processing: C:\Users\L14\Downloads\ABH CSV\ACTIONS\DIG\NO_MASK_total_fusionne.csv


KeyError: 'code_voisin'

In [ ]:
### Generate the files add_data.csv, replace_data.csv, delete_data.csv from actions_digifor.csv
update_files_resume = []

for csv_path in CSVs:
    agent = csv_path.stem
    df_action_dig = pd.read_csv(csv_path, sep=";")
    df_action_dig.index = range(len(df_action_dig))
    
    print("Processing:", path_action_dig)

    ### Generate add_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_ajouter, type_voisin, num_voisin, cni_voisin, signature_voisin, a_signe)
    #print(df_action_dig.columns)
    is_add = df_action_dig["action"].isin(["ADD", "ADD_SEARCH"])
    is_voisin_plle = df_action_dig["type_voisin"].isin(["voisin_plle", "limit_naturelle"])
    is_voisin_riv = (
        (df_action_dig["type_voisin"] == "voisin_riv") &
        (df_action_dig["cni_voisin"].fillna("") != "")
    )

    mask_voisin_plle = (
        (is_add & is_voisin_plle) |
        (is_add & is_voisin_riv)
    )

    df_action_dig.loc[mask_voisin_plle, "nom_a_ajouter"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(":").str[1].str.replace("_"," ").str.strip()
    )
    df_action_dig.loc[mask_voisin_plle, "code_uniq"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(":").str[0].str.strip()
    )

    df_action_dig.loc[mask_voisin_plle, ["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_ajouter","nameOfPersonOriginal", "nameOfNeighborOriginal", "type_voisin", "code_voisin", "num_voisin", "cni_voisin", "position_ctb", "position_dig", "signature_voisin"]].to_csv(f"{out_folder}/{csv_path.stem}_add_data.csv", sep=",", encoding="utf-8-sig")

    ### Generate replace_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_remplacer, nom_a_ajouter, type_voisin, num_voisin, cni_voisin, signature_voisin, a_signe)

    mask_replace_d = df_action_dig["action"] == "REPLACE_D"

    df_action_dig.loc[mask_replace_d, "nom_a_ajouter"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(':', n=1).str[-1].str.replace("_"," ").str.strip()
    )
    df_action_dig.loc[mask_replace_d, "nom_a_remplacer"] = (
        df_action_dig["ancien_nom"].astype(str).str.split("(").str[0].str.replace("_"," ").str.strip()
    )
    df_action_dig.loc[mask_replace_d, "code_uniq"] = (
        df_action_dig["nouveau_nom"].astype(str).str.split(":").str[0].str.strip()
    )

    df_action_dig.loc[mask_replace_d, ["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_remplacer","nameOfPersonOriginal", "nameOfNeighborOriginal", "nom_a_ajouter", "type_voisin", "num_voisin", "cni_voisin", "position_ctb", "position_dig", "signature_voisin"]].to_csv(f"{out_folder}/{csv_path.stem}_replace_data.csv", sep=",", encoding="utf-8-sig")

    ### Generate delete_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_supprimer, type_voisin, a_signe)

    mask_delete = df_action_dig["action"] == "DELETE"

    df_action_dig.loc[mask_delete, "nom_a_supprimer"] = (
        df_action_dig["ancien_nom"].astype(str).str.split("(").str[0].str.replace("_"," ")
    )

    df_action_dig.loc[mask_delete, ["code", "code_parcelle", "village", "date", "nom_a_supprimer","nameOfPersonOriginal", "nameOfNeighborOriginal", "cni_voisin", "position_dig", "type_voisin"]].to_csv(f"{out_folder}/{csv_path.stem}_delete_data.csv", sep=",", encoding="utf-8-sig")

    no_mask = ~ (mask_voisin_plle | mask_replace_d | mask_delete)
    df_action_dig.loc[no_mask, ["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "nameOfPersonOriginal", "nameOfNeighborOriginal", "type_voisin", "action"]].to_csv(f"{out_no_mask_folder}/{csv_path.stem}_nomask_data.csv", sep=";", encoding="utf-8-sig") 

    update_files_resume.append({
        "Agent": agent,
        "Total lignes" : len(df_action_dig),
        "ADD" : mask_voisin_plle.sum(),
        "REPLACE" : mask_replace_d.sum(),
        "DELETE": mask_delete.sum(),
        "NO_MASK" : no_mask.sum(),
        "Somme totale" :
        mask_voisin_plle.sum()
        + mask_replace_d.sum()
        + mask_delete.sum()
        + no_mask.sum()
    })

df_update_files = pd.DataFrame(update_files_resume,
                               columns=["Agent", "Total lignes", "ADD", "REPLACE", "DELETE", "NO_MASK", "Somme totale"])

df_update_files.to_csv(f"resume_data.csv", sep=";", encoding="utf-8-sig")

In [ ]:
### Get informations (cni and signatory) to add in the actions_ctb actions and divide the actions between keep to actions_ctb and add to actions_dig

for path_action_ctb in paths_action_ctb:
    print(path_action_ctb)
    df_voisins_pvcl.loc[:, "code_vois"] = (
        df_voisins_pvcl["village"].astype(str).str.strip()
        + "-"
        + df_voisins_pvcl["nameOfNeighbor"].astype(str).str.strip()
    )

    df_voisins_pvcl = (
        df_voisins_pvcl
        .drop_duplicates("code_vois", keep="first")
    )

    df_voisins_signes.loc[:, "code_vois"] = (
        df_voisins_signes["village"].astype(str).str.strip()
        + "-"
        + df_voisins_signes["nameOfPerson"].astype(str).str.strip()
    )

    df_voisins_signes = (
        df_voisins_signes
        .drop_duplicates("code_vois", keep="first")
    )
    
    df_action_ctb = pd.read_csv(path_action_ctb, sep=";", encoding="utf-8-sig")

    df_action_ctb.loc[:, "village"] = (df_action_ctb["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )
    df_action_ctb.loc[:, "code_vois_old"] = (
        df_action_ctb["village"].str.strip() + "-" +
        df_action_ctb["ancien_nom"].astype(str).str.split(":")
        .str[1].str.split("(")
        .str[0].str.strip()
    )

    df_action_ctb.loc[:, "code_vois_new"] = (
        df_action_ctb["village"].str.strip() + "-" +
        df_action_ctb["nouveau_nom"].astype(str).str.split(":").str[1].str.split("(").str[0].str.strip()
    )

    df_action_ctb = df_action_ctb.merge(
        df_voisins_signes[["code_vois", "numberCNI", "signatoryPhoto"]],
        left_on="code_vois_old",
        right_on="code_vois",
        how="left",
        
        suffixes=("","_autre")
    )

    df_action_ctb.rename(columns={"numberCNI": "cni_voisin_old", "signatoryPhoto": "signature_voisin_old"}, inplace=True)

    df_action_ctb = df_action_ctb.merge(
        df_voisins_signes[["code_vois", "numberCNI","signatoryPhoto"]],
        left_on="code_vois_new",
        right_on="code_vois",
        how="left",
        suffixes=("","_autre")
    )
    
    df_action_ctb.rename(columns={"numberCNI": "cni_voisin_new", "signatoryPhoto": "signature_voisin_new"}, inplace=True)
    df_action_ctb.drop(columns=["code_vois", "code_vois_old", "code_vois_new", "code_vois_autre"], inplace=True)

    # Detect the rows with "ancien_nom" signed then replace nouveau_nom with ancien_nom (add to actions_digifor) (code, code_parcelle, village, date, code_uniq, nom_a_remplacer, nom_a_ajouter, type_voisin, num_voisin, cni_voisin, signature_voisin, a_signe)
    
    mask_old_name = df_action_ctb["cni_voisin_old"].fillna("") != ""
    df_add_action_dig = df_action_ctb.loc[mask_old_name, :].copy()
    df_add_action_dig.loc[:, "nom_a_remplacer"] = (df_add_action_dig.loc[:, "nouveau_nom"].astype(str)
                                                    .str.split(":")
                                                    .str[1]
                                                    .str.split("(signe",regex=False)
                                                    .str[0]
                                                    .str.strip()
    )
    df_add_action_dig.loc[:, "nom_a_ajouter"] = (df_add_action_dig.loc[:, "ancien_nom"].astype(str)
                                                    .str.split(":")
                                                    .str[1]
                                                    .str.strip()
    )
    df_add_action_dig.loc[:, "code_uniq"] = (df_add_action_dig.loc[:, "ancien_nom"].astype(str)
                                                    .str.split(":")
                                                    .str[0]
                                                    .str.strip()
    )
    df_add_action_dig.rename(columns={"cni_voisin_old": "cni_voisin", "signature_voisin_old": "signature_voisin"}, inplace=True)
    
    # Detect the rows with "nouveau_nom" signed and "ancien_nom" not signed then replace ancien_nom with nouveau_nom (keep in actions_ctb) (code	code_parcelle	village	date	indice_unique_voisin	nom_a_remplacer	designation_voisin	type_voisin	cni_voisin	action)
    mask_new_name = (df_action_ctb["cni_voisin_old"].fillna("") == "") & (df_action_ctb["cni_voisin_new"].fillna("") != "")
    df_keep_action_ctb = df_action_ctb.loc[mask_new_name, :].copy()
    df_keep_action_ctb.loc[:, "indice_unique_voisin"] = (df_keep_action_ctb.loc[:, "ancien_nom"].astype(str)
                                                    .str.split(":")
                                                    .str[0]
                                                    .str.strip()
    )
    df_keep_action_ctb.loc[:, "nom_a_remplacer"] = (df_keep_action_ctb.loc[:, "ancien_nom"].astype(str)
                                                    .str.split(":")
                                                    .str[1]
                                                    .str.strip()
    )
    df_keep_action_ctb.loc[:, "designation_voisin"] = (df_keep_action_ctb.loc[:, "nouveau_nom"].astype(str)
                                                    .str.split(":")
                                                    .str[1]
                                                    .str.split("(signe",regex=False)
                                                    .str[0]
                                                    .str.strip()
    )
    
    df_add_action_dig.rename(columns={"cni_voisin_new": "cni_voisin"}, inplace=True)

    # Detect the rows with "nouveau_nom"not signed and "ancien_nom" not signed then manually choose the action
    no_mask_old_new_name = (df_action_ctb["cni_voisin_old"].fillna("") == "") & (df_action_ctb["cni_voisin_new"].fillna("") == "")
    df_manual_action = df_action_ctb.loc[no_mask_old_new_name, :].copy()

    #"code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "action"
    df_add_action_dig.loc[:, "type_voisin"] = ""
    #df_add_action_dig.loc[:, "num_voisin"] = ""
    

    df_add_action_dig[["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_remplacer", "nom_a_ajouter", "type_voisin", "cni_voisin", "signature_voisin"]].to_csv(f"{out_replace_ctb_folder}/{path_action_ctb.stem}_replace_data.csv", sep=";", encoding="utf-8-sig")

    df_keep_action_ctb[["code",	"code_parcelle", "village", "date", "indice_unique_voisin", "nom_a_remplacer", "designation_voisin", "cni_voisin_old", "cni_voisin_new", "action"]].to_csv(f"{out_to_ctb_folder}/{path_action_ctb.stem}.csv", sep=";", encoding="utf-8-sig")

    df_manual_action[["code",	"code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "cni_voisin_old", "cni_voisin_new", "action"]].to_csv(f"{out_ctb_manual_folder}/{path_action_ctb.stem}_manual.csv", sep=";", encoding="utf-8-sig")
    

C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT (1SAVA)_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT ABoudou 05-03_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT BLESSI_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT BOGBEU_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT bossoh_copie_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT Calvin 05-03_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT DIOMANDE_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT Gompou 05-03_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT GUILLAUME_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT HAMID_ctb_actions_ctb.csv
C:\Users\L14\Downloads\ABH CSV\ACTIONS\CTB\ALL_TABLES_EXPORT Hermann_ctb_actions_ctb.csv
C